# Model Training & Hyperparameter Tuning — Bank Account Fraud (BAF)

This notebook trains four fraud detection models with `RandomizedSearchCV` hyperparameter tuning.

**Prerequisites:** Run `python DATA/preprocess.py` first to generate `outputs/processed/`.

**Note on runtime:** Tuning on 1M rows is slow. By default, search is run on a **20% stratified subsample** of the training set to keep it manageable. The best parameters are then used to train the final model on the **full** training set. Increase `TUNE_SAMPLE_FRAC` (up to 1.0) for a more thorough search.

## 1. Imports & Config

In [ ]:
import os
import sys

import joblib
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split

PROCESSED_DIR   = "../outputs/processed"
MODEL_OUT_DIR   = "../outputs"
RANDOM_STATE    = 42
TUNE_SAMPLE_FRAC = 0.20   # fraction of training set used for hyperparameter search
N_ITER           = 10     # number of random parameter combinations to try per model
CV_FOLDS         = 3      # CV folds inside RandomizedSearchCV

os.makedirs(MODEL_OUT_DIR, exist_ok=True)

## 2. Load Processed Data

In [ ]:
if not os.path.exists(PROCESSED_DIR):
    sys.exit("outputs/processed/ not found. Run: python DATA/preprocess.py")

X_train = np.load(f"{PROCESSED_DIR}/X_train.npy", allow_pickle=True)
X_val   = np.load(f"{PROCESSED_DIR}/X_val.npy",   allow_pickle=True)
X_test  = np.load(f"{PROCESSED_DIR}/X_test.npy",  allow_pickle=True)
y_train = np.load(f"{PROCESSED_DIR}/y_train.npy", allow_pickle=True)
y_val   = np.load(f"{PROCESSED_DIR}/y_val.npy",   allow_pickle=True)
y_test  = np.load(f"{PROCESSED_DIR}/y_test.npy",  allow_pickle=True)

neg, pos = np.bincount(y_train.astype(int))
scale_pos_weight = neg / pos

print(f"Train : {len(y_train):>8,}  fraud rate: {y_train.mean():.4f}")
print(f"Val   : {len(y_val):>8,}  fraud rate: {y_val.mean():.4f}")
print(f"Test  : {len(y_test):>8,}  fraud rate: {y_test.mean():.4f}")
print(f"\nscale_pos_weight : {scale_pos_weight:.1f}")

# Stratified subsample for hyperparameter search
X_tune, _, y_tune, _ = train_test_split(
    X_train, y_train,
    train_size=TUNE_SAMPLE_FRAC,
    stratify=y_train,
    random_state=RANDOM_STATE,
)
print(f"\nTuning subsample : {len(y_tune):,}  fraud rate: {y_tune.mean():.4f}")

## 3. Hyperparameter Grids

Edit these distributions to widen or narrow the search space.

In [ ]:
param_grids = {
    "LogisticRegression": {
        "C":      [0.001, 0.01, 0.1, 1, 10, 100],
        "solver": ["lbfgs", "liblinear"],
    },
    "RandomForest": {
        "n_estimators":    [100, 200, 300],
        "max_depth":       [None, 10, 20, 30],
        "min_samples_split": [2, 5, 10],
        "max_features":    ["sqrt", "log2"],
    },
    "LightGBM": {
        "n_estimators":      [200, 500, 1000],
        "learning_rate":     [0.01, 0.05, 0.1],
        "num_leaves":        [15, 31, 63, 127],
        "min_child_samples": [20, 50, 100],
        "subsample":         [0.7, 0.8, 0.9, 1.0],
    },
    "XGBoost": {
        "n_estimators":    [200, 500, 1000],
        "learning_rate":   [0.01, 0.05, 0.1],
        "max_depth":       [4, 6, 8],
        "subsample":       [0.7, 0.8, 0.9, 1.0],
        "colsample_bytree":[0.7, 0.8, 0.9, 1.0],
    },
}

base_models = {
    "LogisticRegression": LogisticRegression(
        class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE
    ),
    "RandomForest": RandomForestClassifier(
        class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE
    ),
    "LightGBM": lgb.LGBMClassifier(
        scale_pos_weight=scale_pos_weight, n_jobs=-1,
        random_state=RANDOM_STATE, verbose=-1
    ),
    "XGBoost": xgb.XGBClassifier(
        scale_pos_weight=scale_pos_weight, eval_metric="aucpr",
        n_jobs=-1, random_state=RANDOM_STATE, verbosity=0
    ),
}

## 4. Hyperparameter Tuning with RandomizedSearchCV

Searches on the tuning subsample, scored by AUPRC (primary KPI for imbalanced fraud detection).

In [ ]:
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
best_params = {}

for name, model in base_models.items():
    print(f"Tuning {name}...")
    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_grids[name],
        n_iter=N_ITER,
        scoring="average_precision",
        cv=cv,
        n_jobs=-1,
        random_state=RANDOM_STATE,
        verbose=0,
        refit=False,  # we'll refit on the full training set below
    )
    search.fit(X_tune, y_tune)
    best_params[name] = search.best_params_
    print(f"  Best CV AUPRC : {search.best_score_:.4f}")
    print(f"  Best params   : {search.best_params_}\n")

## 5. Train Tuned Models on Full Training Set

In [ ]:
def find_best_f1_threshold(y_true, y_proba):
    """Return the decision threshold that maximises F1 on the given set."""
    precision, recall, thresholds = precision_recall_curve(y_true, y_proba)
    f1_scores = np.where(
        (precision + recall) == 0, 0,
        2 * precision * recall / (precision + recall)
    )
    best_idx = np.argmax(f1_scores[:-1])  # last entry has no matching threshold
    return thresholds[best_idx], f1_scores[best_idx]


tuned_models = {}
results = {}

for name, model in base_models.items():
    # Rebuild with best params
    tuned = model.set_params(**best_params[name])
    tuned.fit(X_train, y_train)
    tuned_models[name] = tuned

    val_proba = tuned.predict_proba(X_val)[:, 1]
    val_auprc = average_precision_score(y_val, val_proba)

    # F1 at default threshold (0.5)
    val_pred_05 = (val_proba >= 0.5).astype(int)
    f1_05 = f1_score(y_val, val_pred_05)

    # F1 at optimal threshold (maximises F1 on validation set)
    best_thresh, best_f1 = find_best_f1_threshold(y_val, val_proba)

    results[name] = {
        "model":        tuned,
        "val_auprc":    val_auprc,
        "f1_at_0.5":    f1_05,
        "best_f1":      best_f1,
        "best_thresh":  best_thresh,
    }

print(f"{'Model':<22} {'Val AUPRC':<12} {'F1 @0.5':<12} {'Best F1':<12} {'Threshold'}")
print("-" * 72)
for name, r in results.items():
    print(
        f"{name:<22} {r['val_auprc']:<12.4f} "
        f"{r['f1_at_0.5']:<12.4f} {r['best_f1']:<12.4f} {r['best_thresh']:.4f}"
    )

best_name = max(results, key=lambda k: results[k]["val_auprc"])
print(f"\nBest model by Val AUPRC: {best_name}")

## 6. Best Parameters Summary

In [ ]:
for name, params in best_params.items():
    print(f"{name}:")
    for k, v in params.items():
        print(f"  {k}: {v}")
    print()

## 7. Precision-Recall Curves — Validation Set

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
baseline = y_val.mean()
ax.axhline(baseline, color="gray", linestyle="--",
           label=f"Baseline (prevalence={baseline:.4f})")

for name, r in results.items():
    y_proba = r["model"].predict_proba(X_val)[:, 1]
    precision, recall, _ = precision_recall_curve(y_val, y_proba)
    ax.plot(recall, precision,
            label=f"{name} (AUPRC={r['val_auprc']:.3f}, Best F1={r['best_f1']:.3f})")

ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curves — Validation Set (Tuned Models)")
ax.legend(loc="upper right")
fig.tight_layout()

os.makedirs(f"{MODEL_OUT_DIR}/figures", exist_ok=True)
fig.savefig(f"{MODEL_OUT_DIR}/figures/pr_curves_val.png", dpi=150)
plt.show()

## 8. Save Tuned Models & Best Thresholds

In [ ]:
thresholds = {}

for name, r in results.items():
    model_path = os.path.join(MODEL_OUT_DIR, f"{name}.joblib")
    joblib.dump(r["model"], model_path)
    thresholds[name] = r["best_thresh"]
    print(f"Saved {name} → {model_path}  (best threshold: {r['best_thresh']:.4f})")

joblib.dump(thresholds, os.path.join(MODEL_OUT_DIR, "best_thresholds.joblib"))
print("\nSaved best_thresholds.joblib")
print(f"\nBest model: {best_name}")
print("Run MODELS/evaluate.py for final test-set evaluation.")